In [55]:
# Importations
import os
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import round,col, dayofmonth,month,year,quarter,when 
from pyspark.sql import functions as F

In [56]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [57]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}

In [58]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerSilverTransformation") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

26/05/09 01:32:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [59]:
# Récupère la table weatherforectastapi_raw depuis PostgreSQL dans un DataFrame Spark
weatherforecastapi_raw_df = spark.read.jdbc(
    url=JDBC_URL,
    table='bronze.weatherforecastapi_raw',
    properties=JDBC_PROPS
)

In [60]:
# Nettoyage et enrichissement des données météo
# Transformation: arrondir les colonnes "wind_gusts_10m", "temperature_2m", "cloud_cover"
# Transformation: créer la colonne "day" à partir de "date" avec la méthode "dayofmonth"
# Transformation: créer la colonne "month" à partir de "date" avec la méthode "month"
# Transformation: créer la colonne "quarter" à partir de "date" avec la méthode "quarter"
# Transformation: créer la colonne "year" à partir de "date" avec la méthode "year"
# Transformation: formater la colonne "time" au format "HH:mm:ss"
# Transformation: créer les colonnes "hour_of_day", "minute_of_hour", "second_of_minute" à partir de "time"
# Transformation: créer la colonne "time_period" (Morning, Afternoon, Evening, Night) à partir de "hour_of_day"

df_transformed = (
    weatherforecastapi_raw_df
    .withColumn("wind_gusts_10m", round(col("wind_gusts_10m"), 2))
    .withColumn("temperature_2m", round(col("temperature_2m"), 2))
    .withColumn("cloud_cover", round(col("cloud_cover"), 2))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("year", year(col("date")))
    .withColumn("time", F.date_format(col("time"), "HH:mm:ss"))
    .withColumn("hour_of_day", F.hour(col("time")))
    .withColumn("minute_of_hour", F.minute(col("time")))
    .withColumn("second_of_minute", F.second(col("time")))
    .withColumn(
        "time_period",
        when((col("hour_of_day") >= 5) & (col("hour_of_day") < 12), "Morning")
        .when((col("hour_of_day") >= 12) & (col("hour_of_day") < 17), "Afternoon")
        .when((col("hour_of_day") >= 17) & (col("hour_of_day") < 21), "Evening")
        .otherwise("Night")
    )
)

# Suppression des colonnes des métadonnées
df_transformed = df_transformed.drop("ingested_at","source_api")

df_transformed.show(10)

+----------+--------+---------+-----------+--------+--------------------+--------------+--------------+-----------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|      date|    time| latitude|  longitude|  region|         region_name|wind_gusts_10m|temperature_2m|cloud_cover|day|month|quarter|year|hour_of_day|minute_of_hour|second_of_minute|time_period|
+----------+--------+---------+-----------+--------+--------------------+--------------+--------------+-----------+---+-----+-------+----+-----------+--------------+----------------+-----------+
|2024-06-15|00:00:00|34.059753|-118.237503|Region A|Los Angeles, Cali...|         10.80|         17.55|       0.00| 15|    6|      2|2024|          0|             0|               0|      Night|
|2024-06-15|00:00:00|36.801403|-119.448090|Region B|Fresno/Central Va...|         10.44|         23.70|       0.00| 15|    6|      2|2024|          0|             0|               0|      Night|
|2024-06-15|00:00:00|40.7

In [61]:
# Définition de la table silver.weatherforecastapi_cleaned avec les types de données appropriés
create_table_sql = """
CREATE TABLE IF NOT EXISTS silver.weatherforecastapi_cleaned (
    weather_id BIGSERIAL PRIMARY KEY,
    date DATE,
    time TIME,
    latitude NUMERIC(9,6),
    longitude NUMERIC(9,6),
    region VARCHAR(100),
    region_name VARCHAR(255),
    wind_gusts_10m NUMERIC(6,2),
    temperature_2m NUMERIC(5,2),
    cloud_cover NUMERIC(5,2),
    day INTEGER,
    month INTEGER,
    quarter INTEGER,
    year INTEGER,
    hour_of_day INTEGER,
    minute_of_hour INTEGER,
    second_of_minute INTEGER,
    time_period VARCHAR(20),
    CONSTRAINT unique_weather_record UNIQUE (date, time, region, region_name)
);
"""

# Connexion à PostgreSQL et exécution du script de création de table
with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        # Crée le schéma s'il n'existe pas
        cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
        
        # Crée la table avec script SQL
        cur.execute(create_table_sql)
        print("Schéma silver et table weatherforecastapi_cleaned créés avec les types corrects")
        
        # Ajouter la contrainte UNIQUE métier si elle n'existe pas
        try:
            cur.execute("""
                ALTER TABLE silver.weatherforecastapi_cleaned 
                ADD CONSTRAINT unique_weather_record UNIQUE (date, time, region, region_name);
            """)
            print("Contrainte UNIQUE ajoutée sur (date, time, region, region_name)")
        except (psycopg.errors.DuplicateTable, psycopg.errors.DuplicateObject):
            print("Contrainte UNIQUE existe déjà")
        

Schéma silver et table weatherforecastapi_cleaned créés avec les types corrects
Contrainte UNIQUE existe déjà


In [ ]:
# Insertion des données dans la table en mode UPSERT

total_count = df_transformed.count()
print(f"Lignes à traiter: {total_count}")

# Convertir le DataFrame Spark en Pandas pour l'insertion avec psycopg
df_transformed_pd = df_transformed.toPandas()

upsert_sql = """
    INSERT INTO silver.weatherforecastapi_cleaned (
        date, time, latitude, longitude, region, region_name, 
        wind_gusts_10m, temperature_2m, cloud_cover, day, month, quarter, year,
        hour_of_day, minute_of_hour, second_of_minute, time_period
    ) VALUES (%(date)s, %(time)s, %(latitude)s, %(longitude)s, %(region)s, %(region_name)s,
            %(wind_gusts_10m)s, %(temperature_2m)s, %(cloud_cover)s, %(day)s, %(month)s, %(quarter)s, %(year)s, %(hour_of_day)s, %(minute_of_hour)s, %(second_of_minute)s, %(time_period)s)
    ON CONFLICT (date, time, region, region_name) DO UPDATE SET
        latitude = EXCLUDED.latitude,
        longitude = EXCLUDED.longitude,
        wind_gusts_10m = EXCLUDED.wind_gusts_10m,
        temperature_2m = EXCLUDED.temperature_2m,
        cloud_cover = EXCLUDED.cloud_cover,
        day = EXCLUDED.day,
        month = EXCLUDED.month,
        quarter = EXCLUDED.quarter,
        year = EXCLUDED.year,
        hour_of_day = EXCLUDED.hour_of_day,
        minute_of_hour = EXCLUDED.minute_of_hour,
        second_of_minute = EXCLUDED.second_of_minute,
        time_period = EXCLUDED.time_period
    RETURNING (xmax = 0) AS inserted
"""

new_rows_count:int = 0
duplicates_avoided_count:int = 0

with psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    autocommit=True
) as conn:
    with conn.cursor() as cur:
        for _, row in df_transformed_pd.iterrows():
            params = {str(k): v for k, v in row.items()}
            cur.execute(upsert_sql, params)
            result = cur.fetchone() # Récupération du résultat de l'UPSERT
            inserted = bool(result[0]) if result is not None else False # Vérifie si la ligne a été insérée ou mise à jour
            if inserted:
                new_rows_count += 1
            else:
                duplicates_avoided_count += 1

print(f"{total_count} lignes traitées avec UPSERT")
print(f"Nouvelles lignes: {new_rows_count}")
print(f"Doublons évités: {duplicates_avoided_count}")

Lignes à traiter: 432
432 lignes traitées avec UPSERT
Nouvelles lignes: 432
Doublons évités: 0


In [63]:
# Arrêt de la session Spark
spark.stop()